In [ ]:
#| hide
from lekhaka.verses import *
from lekhaka.repairs import *
from lekhaka.editions import *
from lekhaka.book import *

# lekhaka

> Books of verses read into chapters: verse numbers mended, chapters assembled, akṣaras repaired. On vishalakshi.

lekhaka (लेखक, a scribe) turns a printed book of verses into its chapters. Each page is read by a vision model through [vishalakshi](https://github.com/vedicreader/vishalakshi), local or hosted, as blocks: verses with their printed numbers, headings, colophons. lekhaka gathers the verses into chapters, mends the numbering slips a reader makes, repairs malformed Devanagari akṣaras, and lists what a proofreader should check.

## Install

```sh
pip install lekhaka
```

## How to use

```python
from lekhaka import read_book, book_md
from vishalakshi import Vault

b = read_book('gita.pdf', Vault('books.db'), model='claude/sonnet', title='Bhagavad Gita',
              hint='Each verse ends in ॥ n ॥.', counts=[47, 72])
b.chapters[0]['verses'][:2]     # [{n, text, page, heading}, ...]
b.probs                         # numbers mended, counts that differ, for a proofreader
print(book_md(b.chapters, 'Bhagavad Gita'))
```

With a vault the pages are kept as versions of one document: `vault.proofread(b.doc_id, readers)` has other models read them again, and `vault.accept` applies the changes the owner chooses. Without a vault nothing is stored.

The pieces work on their own, on the pages `vishalakshi.read_doc` returns or any `{page, blocks}` list:

## On real books

`nbs/samples` holds a few pages each of three books vedicreader has read, as a model returned them. Sixteen pages of Varāhamihira's Bṛhajjātaka assemble into its third and fourth chapters, each opened by its heading and closed by its colophon:

In [ ]:
import json
bj = json.load(open('samples/brihat_jataka.json'))
chs, probs = assemble(bj)
[(c['heading'], len(c['verses']), (c['colophon'] or '')[:40]) for c in chs]

[('वियोनिजन्माध्यायः', 8, 'इति श्रीवराहमिहिरकृते बृहज्जातके वियोनिज'),
 ('निषेकाध्यायः', 22, 'इति श्रीवराहमिहिरकृते बृहज्जातके निषेकाध')]

In [ ]:
probs

[]

Its colophons number the chapters, so `assemble_colophon` can cut there instead, reading each chapter's number and name from its colophon; `cut` lays the chapters into titles and `numbering` reports a run's numbers:

In [ ]:
cc, _ = assemble_colophon(bj)
[(c['part'], c['no'], c['name'], len(c['verses'])) for c in cc]

[('पूर्व', 3, 'वियोनिजन्म', 8), ('पूर्व', 4, 'निषेक', 22)]

In [ ]:
[t['title'] for t in cut(cc, 'chapters', book='बृहज्जातकम्')], numbering(cc[1]['verses'])

(['बृहज्जातकम् १ वियोनिजन्म', 'बृहज्जातकम् २ निषेक'],
 {'numbered': 22,
  'unnumbered': 0,
  'skipped': [],
  'repeated': [],
  'first': 1,
  'last': 22})

The Vidhānamālai prints each verse's number at the end of its urai, the prose gloss after it. The urai goes into its verse's notes, and gives the verse its number:

In [ ]:
vm = json.load(open('samples/vidhanamalai.json'))
chs, probs = assemble(vm, notes=('urai',), number_from_note=True)
v = chs[0]['verses'][0]
len(chs[0]['verses']), v['n'], v['text'].split('\n')[0], v['notes']['urai'][0][:60]

(11,
 1,
 'பூமகள் கேள்வன்பொன் னுடையன் புட்கொடி யோன் புனித,',
 '(என்றது.) ஸ்ரீமகாலக்ஷ்மிக்கு நாயகரும், பீதாம்பரத்தையுடையவரும')

Uromarishi's Pañcapakṣi (1917) numbers its verses in Tamil numerals with the old signs for ten and a hundred:

In [ ]:
uro = json.load(open('samples/uromarishi.json'))
[(b['n'], vnum(b['n'])) for p in uro for b in p['blocks'] if '௱' in str(b.get('n') or '')][:4]

[('௨௱௫', 205), ('௨௱௬', 206), ('௨௱௭', 207), ('௨௱௮', 208)]

## The pieces

- `vnum`, `mend`, `renumber`, `assemble`, `assemble_colophon`, `check_counts`: numbers and chapters. A page's blocks may sit under `blocks` or `items`; a `head` is a heading (`KINDS`); `join_cont` rejoins a block a page break cut.
- `apply_fixes`, `number_runs`, `borrow_numbers`, `cut_by_plan`, `drop_pages`: what a reading gets wrong, put right before `assemble`. `fix_chapters`, `sutras`: after it.
- `fix_aksharas`, `bad_aksharas`, `norm`, `virama`: malformed akṣaras and typing slips in Devanagari.
- `cut`, `numbering`: verses laid into titles. `witness_verses`, `text_changes`, `ref_check`, `move_lines`: a reading checked against another edition. `verse_text`, `disagree`, `compare`: two readings of a page compared.
- `read_book`, `book_md`: a book end to end, through vishalakshi.

In [ ]:
fix_aksharas('ध्ार्म'), vnum('२६॥२७'), vnum('२६॥ १', pick='first')

('धार्म', 27, 26)